In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import json


BASE_URL = "https://"

OUTPUT_FILE = "oferty_najmu.xlsx"
RESULT_FILE = "oferty_najmu_WYNIKI.xlsx"

In [4]:
#funkcja pobierająca dane
def get_page_data(page):

    url = f"{BASE_URL}?strona={page}"

    response = requests.get(url)

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    cards = soup.find_all(
        "article",
        class_="list__item"
    )

    listings = []

    for card in cards:

        link = card.find(
            "a",
            href=re.compile(r"-ogl\d+\.html$")
        )

        if link is None:
            continue

        offer_url = link["href"]

        text = card.get_text(
            " ",
            strip=True
        )

        # Tytuł
        title = None

        heading = card.find(
            ["h2", "h3", "h4"]
        )

        if heading:
            title = heading.get_text(
                " ",
                strip=True
            )

        # Cena
        price = None

        match = re.search(
            r"(\d[\d\s]*)\s*zł",
            text
        )

        if match:
            price = int(
                match.group(1).replace(" ", "")
            )

        # Metraż
        area_m2 = None

        match = re.search(
            r"(\d+(?:[.,]\d+)?)\s*m\s*2",
            text
        )

        if match:
            area_m2 = float(
                match.group(1).replace(",", ".")
            )

        # Liczba pokoi
        rooms = None

        match = re.search(
            r"(\d+)\s+(?:pokój|pokoje)",
            text
        )

        if match:
            rooms = int(match.group(1))

        # Piętro
        floor = None

        if "parter" in text.lower():

            floor = 0

        else:

            match = re.search(
                r"(\d+)\s+piętro",
                text
            )

            if match:
                floor = int(match.group(1))

        # Rok budowy
        year_built = None

        match = re.search(
            r"(\d{4})\s*r\.",
            text
        )

        if match:
            year_built = int(match.group(1))

        listings.append({
            "title": title,
            "price": price,
            "area_m2": area_m2,
            "rooms": rooms,
            "floor": floor,
            "year_built": year_built,
            "url": offer_url
        })

    df = pd.DataFrame(listings)

    df = df.drop_duplicates(
        subset="url"
    )

    return df

In [5]:
#pobieranie ofert
df_all = pd.DataFrame()

page = 1

while True:

    print(f"Pobieram stronę {page}...")

    try:
        df_page = get_page_data(page)

    except Exception as e:
        print(f"Błąd: {e}")
        break

    print(f"  znaleziono: {len(df_page)} ofert")

    if df_page.empty:
        print("  Brak ofert - koniec.")
        break

    df_all = pd.concat(
        [df_all, df_page],
        ignore_index=True
    )

    print(f"  razem: {len(df_all)}")

    # zapis po każdej stronie
    df_all.to_excel(
        OUTPUT_FILE,
        index=False
    )

    page += 1

    time.sleep(0.7)


print()
print("=" * 50)
print("GOTOWE")
print("=" * 50)
print(f"Liczba stron: {page - 1}")
print(f"Liczba ofert: {len(df_all)}")

Pobieram stronę 1...
  znaleziono: 30 ofert
  razem: 30
Pobieram stronę 2...
  znaleziono: 30 ofert
  razem: 60
Pobieram stronę 3...
  znaleziono: 30 ofert
  razem: 90
Pobieram stronę 4...
  znaleziono: 30 ofert
  razem: 120
Pobieram stronę 5...
  znaleziono: 30 ofert
  razem: 150
Pobieram stronę 6...
  znaleziono: 30 ofert
  razem: 180
Pobieram stronę 7...
  znaleziono: 30 ofert
  razem: 210
Pobieram stronę 8...
  znaleziono: 30 ofert
  razem: 240
Pobieram stronę 9...
  znaleziono: 30 ofert
  razem: 270
Pobieram stronę 10...
  znaleziono: 30 ofert
  razem: 300
Pobieram stronę 11...
  znaleziono: 30 ofert
  razem: 330
Pobieram stronę 12...
  znaleziono: 30 ofert
  razem: 360
Pobieram stronę 13...
  znaleziono: 30 ofert
  razem: 390
Pobieram stronę 14...
  znaleziono: 30 ofert
  razem: 420
Pobieram stronę 15...
  znaleziono: 30 ofert
  razem: 450
Pobieram stronę 16...
  znaleziono: 30 ofert
  razem: 480
Pobieram stronę 17...
  znaleziono: 30 ofert
  razem: 510
Pobieram stronę 18...
  zn

In [6]:
#Filtr
MAX_PRICE = 5000
MIN_AREA = 50
MIN_ROOMS = 3
MAX_ROOMS = 4

In [7]:
filtered_df = df_all[
    (df_all["price"] <= MAX_PRICE) &
    (df_all["area_m2"] >= MIN_AREA) &
    (df_all["rooms"] >= MIN_ROOMS) &
    (df_all["rooms"] <= MAX_ROOMS)
].copy()

print("Wszystkich ofert:", len(df_all))
print("Po pierwszym filtrze:", len(filtered_df))

Wszystkich ofert: 3403
Po pierwszym filtrze: 653


In [8]:
#Funkcja szczegóły
def get_offer_details(url):

    response = requests.get(url)

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    text = soup.get_text(
        " ",
        strip=True
    ).lower()

    # JSON-LD
    title = None
    price = None

    for script in soup.find_all(
        "script",
        type="application/ld+json"
    ):

        try:

            data = json.loads(
                script.string
            )

            if isinstance(data, dict):

                if not title:
                    title = data.get("name")

                offers = data.get("offers")

                if isinstance(offers, dict):
                    price = offers.get("price")

        except:
            pass

    # Cechy
    balcony = (
        "balkon" in text
        and "bez balkonu" not in text
        and "brak balkonu" not in text
    )

    cellar = (
        "piwnica" in text
        or "komórka lokatorska" in text
    )

    parking = (
        "miejsce parkingowe" in text
        or "miejsce postojowe" in text
        or "garaż" in text
        or "hala garażowa" in text
    )

    pets = (
        "zwierzęta mile widziane" in text
        or "zwierzęta dozwolone" in text
        or "z pupilami" in text
    )

    furnished = (
        "umeblowane" in text
        or "umeblowany" in text
        or "w pełni wyposażone" in text
    )

    # Nowe / po remoncie
    new_or_renovated = (
        "pierwszy najem" in text
        or "pierwszy wynajem" in text
        or "nigdy niezamieszkałe" in text
        or "nigdy nie zamieszkałe" in text
        or "nowe mieszkanie" in text
        or "nowo wybudowane" in text
        or "świeżo po remoncie" in text
        or "świeżo wyremontowane" in text
        or "po generalnym remoncie" in text
        or "generalnym remoncie" in text
    )

    # SKM
    near_skm = (
        "skm" in text
        or "szybka kolej miejska" in text
    )

    return {
        "title": title,
        "price_detail": price,
        "new_or_renovated": new_or_renovated,
        "near_skm": near_skm,
        "balcony": balcony,
        "cellar": cellar,
        "parking": parking,
        "pets": pets,
        "furnished": furnished,
        "url": url
    }

In [9]:
details = []

total = len(filtered_df)

for i, (_, row) in enumerate(
    filtered_df.iterrows(),
    start=1
):

    print(f"Pobieram {i}/{total}...")

    try:

        result = get_offer_details(
            row["url"]
        )

        details.append(result)

    except Exception as e:

        print(f"  BŁĄD: {e}")

        details.append({
            "title": None,
            "price_detail": None,
            "new_or_renovated": None,
            "near_skm": None,
            "balcony": None,
            "cellar": None,
            "parking": None,
            "pets": None,
            "furnished": None,
            "url": row["url"]
        })

    time.sleep(0.5)


details_df = pd.DataFrame(details)

print()
print("Gotowe!")
print(f"Pobrano szczegóły: {len(details_df)}")

Pobieram 1/653...
Pobieram 2/653...
Pobieram 3/653...
Pobieram 4/653...
Pobieram 5/653...
Pobieram 6/653...
Pobieram 7/653...
Pobieram 8/653...
Pobieram 9/653...
Pobieram 10/653...
Pobieram 11/653...
Pobieram 12/653...
Pobieram 13/653...
Pobieram 14/653...
Pobieram 15/653...
Pobieram 16/653...
Pobieram 17/653...
Pobieram 18/653...
Pobieram 19/653...
Pobieram 20/653...
Pobieram 21/653...
Pobieram 22/653...
Pobieram 23/653...
Pobieram 24/653...
Pobieram 25/653...
Pobieram 26/653...
Pobieram 27/653...
Pobieram 28/653...
Pobieram 29/653...
Pobieram 30/653...
Pobieram 31/653...
Pobieram 32/653...
Pobieram 33/653...
Pobieram 34/653...
Pobieram 35/653...
Pobieram 36/653...
Pobieram 37/653...
Pobieram 38/653...
Pobieram 39/653...
Pobieram 40/653...
Pobieram 41/653...
Pobieram 42/653...
Pobieram 43/653...
Pobieram 44/653...
Pobieram 45/653...
Pobieram 46/653...
Pobieram 47/653...
Pobieram 48/653...
Pobieram 49/653...
Pobieram 50/653...
Pobieram 51/653...
Pobieram 52/653...
Pobieram 53/653...
Po

In [10]:
#filtr + generowanie excela 
final_df = filtered_df.merge(
    details_df.drop(
        columns=["title", "price_detail"]
    ),
    on="url",
    how="left"
)


wanted_df = final_df[
    (final_df["rooms"] == 3) &
    (final_df["new_or_renovated"] == True) &
    (final_df["near_skm"] == True)
].copy()


print()
print("=" * 50)
print("WYNIK")
print("=" * 50)
print(f"Liczba ofert spełniających wszystkie kryteria: {len(wanted_df)}")


display(
    wanted_df[
        [
            "title",
            "price",
            "area_m2",
            "floor",
            "new_or_renovated",
            "near_skm",
            "balcony",
            "parking",
            "cellar",
            "url"
        ]
    ]
)


wanted_df.to_excel(
    RESULT_FILE,
    index=False
)

print()
print(f"Zapisano: {RESULT_FILE}")


WYNIK
Liczba ofert spełniających wszystkie kryteria: 41


,title,price,area_m2,floor,new_or_renovated,near_skm,balcony,parking,cellar,url
3,"Piękny Apartament - 3 pokoje, Ogród, Garaż, Sh...",3500.0,66.00,0.0,True,True,False,True,True,https://ogloszenia.trojmiasto.pl/nieruchomosci...
14,"Oliwa - 2 sypialnie + salon, blisko OBC, tramw...",4200.0,61.00,2.0,True,True,True,False,False,https://ogloszenia.trojmiasto.pl/nieruchomosci...
34,"Do wynajęcia 3 pokojowe mieszkanie 55,6 m² po ...",3200.0,55.00,0.0,True,True,False,False,False,https://ogloszenia.trojmiasto.pl/nieruchomosci...
35,"Wynajmę ciche, trzypokojowe mieszkanie w dosko...",4200.0,64.00,4.0,True,True,True,True,True,https://ogloszenia.trojmiasto.pl/nieruchomosci...
66,Nowe 3 pokojowe mieszkanie Kowale osiedle Szum...,3000.0,53.00,1.0,True,True,False,True,False,https://ogloszenia.trojmiasto.pl/nieruchomosci...
73,Komfortowe Mieszkanie | Duży Taras | Widok na ...,3800.0,104.00,1.0,True,True,True,True,True,https://ogloszenia.trojmiasto.pl/nieruchomosci...
97,Wynajem mieszkanie 3 pok Porębskiego 20 Gdańsk,2500.0,55.00,3.0,True,True,True,False,True,https://ogloszenia.trojmiasto.pl/nieruchomosci...
122,"Nowe mieszkanie 53 m2, Gdańsk Śródmieście 3 po...",4000.0,53.00,4.0,True,True,True,True,True,https://ogloszenia.trojmiasto.pl/nieruchomosci...
145,Do wynajęcia 3 pokoje - 350m do Bulwaru Nadm. !!!,3700.0,56.00,1.0,True,True,False,False,False,https://ogloszenia.trojmiasto.pl/nieruchomosci...
170,Świetna lokalizacja,3000.0,53.00,0.0,True,True,True,True,True,https://ogloszenia.trojmiasto.pl/nieruchomosci...



Zapisano: oferty_najmu_WYNIKI.xlsx
